In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import requests
import time
from urllib.parse import urlparse, parse_qs
from IPython.display import HTML, display

In [3]:
def get_all_comments_using_product_url(product_url):
    parsed_url = urlparse(product_url)
    query_params = parse_qs(parsed_url.query)

    if "product_id" in query_params:
        product_id = query_params["product_id"][0]
    else:
        path = parsed_url.path

        if "dkp-" in path:
            product_id = path.split("dkp-")[1].split("/")[0]
        else:
            raise ValueError("Product ID در URL پیدا نشد.")

    print(f"Product ID: {product_id}")

    comments_list = []
    page = 1

    session = requests.Session()

    while True:
        api_url = (
            f"https://api.digikala.com/v1/rate-review/"
            f"products/{product_id}/?page={page}"
        )

        try:
            response = session.get(
                api_url,
                timeout=15
            )
            response.raise_for_status()
            data = response.json()
            comments = data.get("data", {}).get("comments", [])

        except requests.RequestException as e:
            print(f"Error on page {page}: {e}")
            break

        except ValueError:
            print(f"Invalid JSON response on page {page}")
            break

        if not comments:
            break

        for comment in comments:
            comment_id = comment.get("id")
            comment_body = comment.get("body")

            comments_list.append({
                "comment_id": comment_id,
                "comment_body": comment_body
            })
        page += 1
        time.sleep(0.2)
    
    comments = [
    comment for comment in comments_list
    if comment.get("comment_body", "").strip()
    ]

    print(f"\nTotal comments: {len(comments)}")
    
    reviews_json = json.dumps(
    comments,
    ensure_ascii=False,
    indent=2
    )
    return reviews_json

In [4]:
product_name = "دسته بازی پلی استیشن ۴ مدل PS5"
product_category = "Game Controller"
current_price = "3,600,000 تومان"

reviews_json = get_all_comments_using_product_url("https://www.digikala.com/product/dkp-17796220/%D8%AF%D8%B3%D8%AA%D9%87-%D8%A8%D8%A7%D8%B2%DB%8C-%D9%BE%D9%84%DB%8C-%D8%A7%D8%B3%D8%AA%DB%8C%D8%B4%D9%86-%DB%B4-%D9%85%D8%AF%D9%84-ps5/?product_id=17796220&variant_id=80553336")

Product ID: 17796220

Total comments: 108


In [5]:
system_prompt = """
You are a product review analysis system.

You will receive customer reviews of one product.

The reviews may be written in Persian (Farsi).

Your task is ONLY to extract product-related findings from the reviews.

IMPORTANT RULES:

1. Understand Persian reviews directly.
2. Do NOT translate the reviews.
3. Do NOT rewrite the reviews.
4. Do NOT reproduce any review text.
5. Do NOT return the original comments.
6. Do NOT create a rating or score for the product.
7. Do NOT give a purchase recommendation.
8. Do NOT invent information.
9. Ignore empty reviews.
10. Only report information that is supported by the reviews.
11. If something is mentioned only once, do not describe it as a common problem.
12. If multiple reviews mention the same issue, identify it as a recurring issue.
13. The output language must be English.
14. Return ONLY valid JSON.
15. Do not use Markdown.

Extract:

- Positive aspects
- Negative aspects
- Product quality observations
- Performance observations
- Features mentioned by customers
- Price/value opinions
- Recurring problems
- Important considerations for buyers

For every finding, include the IDs of the reviews that support that finding.

Use exactly this JSON structure:

{
  "positive_aspects": [
    {
      "aspect": "",
      "source_comment_ids": [],
      "mentions": 0
    }
  ],

  "negative_aspects": [
    {
      "aspect": "",
      "source_comment_ids": [],
      "mentions": 0,
      "severity": "LOW | MEDIUM | HIGH"
    }
  ],

  "quality_observations": [],

  "performance_observations": [],

  "features": [],

  "price_value_observations": [],

  "recurring_problems": [
    {
      "problem": "",
      "source_comment_ids": [],
      "mentions": 0
    }
  ],

  "buyer_considerations": []
}
"""

In [6]:
def make_user_prompt(product_name, product_category, current_price, reviews_json):
    user_prompt = f"""
    Analyze the following customer reviews.

    The reviews are written in Persian.

    Understand the Persian text directly.

    Do NOT translate, rewrite, or reproduce the reviews.

    Do NOT return the original review text.

    Extract only the product-related findings requested by the system prompt.

    Product:
    {product_name}

    Category:
    {product_category}

    Price:
    {current_price}

    Reviews:

    {reviews_json}
    """
    return user_prompt

In [20]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [27]:
def extracting_info_from_reviews(client, model, system_prompts, reviews_json):
    print(f"Extracting information from the reviews by {model}...\n")
    response = client.chat.completions.create(model=model, messages=[
    {"role": "system", "content": system_prompts[0]},
    {"role": "user", "content": make_user_prompt(product_name, product_category, current_price, reviews_json)}
    ])

    return response.choices[0].message.content
# display(Markdown(response.choices[0].message.content))

In [55]:
raw_output = response.choices[0].message.content

def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.lower().startswith("json"):
            text = text[4:]
    return text.strip()

analysis_json_str = extract_json(raw_output)

try:
    analysis_data = json.loads(analysis_json_str)
    print("valid json in the firts stage✅")
except json.JSONDecodeError as e:
    print("not valid json in the firts stage:", e)
    analysis_data = None


valid json in the firts stage✅


In [13]:
decision_system_prompt = """
You are a product analysis assistant that turns structured review-analysis
data (JSON) into a clear, human-readable buyer's summary.

You will receive a JSON object containing extracted findings from customer
reviews of a product (positive aspects, negative aspects, quality/performance
observations, recurring problems, price/value opinions, buyer considerations).

RULES:
1. Base every statement ONLY on the provided JSON data. Do not invent facts.
2. Weigh findings by their "mentions" count and "severity" (for negative
   aspects) when deciding what to emphasize.
3. Identify patterns: e.g. if a HIGH severity issue has many mentions,
   flag it as a significant risk.
4. Distinguish between use cases based on performance_observations
   and buyer_considerations.
5. Do not give a numeric rating or score.
6. Do not tell the user definitively whether to buy or not — instead,
   explain trade-offs and let the reader decide, tailored to different
   buyer intents if relevant.
7. Output in Persian (Farsi), in clear prose (not JSON), organized under
   short headers.
8. Keep it concise: a short overview, key strengths, key risks/concerns,
   and "who this product is/isn't a good fit for".
"""

In [14]:
def make_decision_user_prompt(product_name, product_category, current_price, analysis_json):
    return f"""
    Based on the following structured review analysis, write a buyer's
    summary for this product.

    Product: {product_name}
    Category: {product_category}
    Price: {current_price}

    Structured findings (JSON):
    {json.dumps(analysis_json, ensure_ascii=False, indent=2)}
    """

In [28]:
def generating_final_suugestion_based_on_the_extracted_info(client, model, system_prompts, analysis_data):
    print(f"Generating final suggestion based on the extracted information by {model}...\n")
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompts[1]},
            {"role": "user", "content": make_decision_user_prompt(
                product_name, product_category, current_price, analysis_data
            )}
        ]
    )

    return response.choices[0].message.content
# display(Markdown(final_response.choices[0].message.content))

In [26]:
system_prompts = [system_prompt, decision_system_prompt]

In [30]:
def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.lower().startswith("json"):
            text = text[4:]
    return text.strip()

In [ ]:
#ALL IN ONE
MODEL_OLLAMA = "llama3.2"
MODEL_GAP = "gapgpt-qwen-3.5"
gap_api_key = os.getenv('GAP_FERMIN_KEY')


def suggest_based_on_the_comments():
    product_url = input("Enter the URL of the product here: ")
    product_reviews_json = get_all_comments_using_product_url(product_url)

    client = OpenAI(base_url='https://api.gapgpt.app/v1', api_key=gap_api_key)
    raw_extracted_info = extracting_info_from_reviews(client, MODEL_GAP, system_prompts, product_reviews_json)
    analyzed_json_str = extract_json(raw_extracted_info)

    try:
        analysis_data = json.loads(analyzed_json_str)
    except json.JSONDecodeError as e:
        print("Invalid json in the firts stage:", e)
        analysis_data = None

    final_suggestion = generating_final_suugestion_based_on_the_extracted_info(client, MODEL_GAP, system_prompts, analysis_data)
    display(HTML(
    f'<div dir="rtl" style="text-align:right; font-family:Tahoma,Arial; '
    f'line-height:1.8; white-space:pre-wrap; unicode-bidi:embed;">{final_suggestion}</div>'
))

In [35]:
def main():
    suggest_based_on_the_comments()

In [36]:
main()

Product ID: 16241000

Total comments: 30
Extracting information from the reviews by gapgpt-qwen-3.5...

Generating final suggestion based on the extracted information by gapgpt-qwen-3.5...

